# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
df = pd.read_csv('data/AviationData.csv', encoding='latin-1', low_memory=False)
print(df.shape)
df.head()


(88889, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [3]:
df['Event.Date'] = pd.to_datetime(df['Event.Date'])
print(df['Event.Date'].min(), '->', df['Event.Date'].max())

1948-10-24 00:00:00 -> 2022-12-29 00:00:00


In [4]:
#### Aircraft.Category - imputing before filtering
1. Placeholder values (Unknown, UNK) -> NaN.
2. If a Make + Model appears elsewhere with a known category, use that.
3. Otherwise, if a Make is almost always (>=95%) one category, use that.
4. Otherwise, if the engine type is fixed-wing and engines >= 1, call it an Airplane.
5. Anything still unknown is dropped.

SyntaxError: invalid syntax (817819639.py, line 2)

In [5]:
df['Aircraft.Category'] = df['Aircraft.Category'].replace({'Unknown': np.nan, 'UNK': np.nan})

# helper keys (upper-cased so 'Cessna' and 'CESSNA' match)
df['_make'] = df['Make'].str.upper().str.strip()
df['_key'] = df['_make'] + '|' + df['Model'].str.upper().str.strip()

known = df.dropna(subset=['Aircraft.Category'])

# step 2: Make+Model lookup
model_cat = known.groupby('_key')['Aircraft.Category'].agg(lambda s: s.mode().iloc[0])
df['Aircraft.Category'] = df['Aircraft.Category'].fillna(df['_key'].map(model_cat))

# step 3: Make lookup, only when the make is >=95% a single category (and has >=5 labelled rows)
known = df.dropna(subset=['Aircraft.Category'])
make_stats = known.groupby('_make')['Aircraft.Category'].agg(
    top=lambda s: s.mode().iloc[0],
    purity=lambda s: s.value_counts(normalize=True).iloc[0],
    n='size')
make_cat = make_stats[(make_stats['purity'] >= 0.95) & (make_stats['n'] >= 5)]['top']
df['Aircraft.Category'] = df['Aircraft.Category'].fillna(df['_make'].map(make_cat))

# step 4: engine-type heuristic for what is left
fixed_wing = df['Engine.Type'].isin(['Reciprocating', 'Turbo Prop', 'Turbo Fan', 'Turbo Jet']) & (df['Number.of.Engines'] >= 1)
still_missing = df['Aircraft.Category'].isna()
df.loc[still_missing & fixed_wing, 'Aircraft.Category'] = 'Airplane'

# a few records categorised via the lookups are really helicopters (Turbo Shaft engines) -> drop them
df = df[df['Engine.Type'] != 'Turbo Shaft'].copy()

print('Category still missing:', df['Aircraft.Category'].isna().sum())
df['Aircraft.Category'].value_counts(dropna=False)

Category still missing: 589


Aircraft.Category
Airplane             77905
Helicopter            4571
Glider                1094
NaN                    589
Balloon                586
Gyrocraft              226
Weight-Shift           163
Powered Parachute       92
Ultralight              32
WSFT                     9
Powered-Lift             6
Blimp                    5
Rocket                   1
ULTR                     1
Name: count, dtype: int64

In [6]:
df = df[df['Aircraft.Category'] == 'Airplane'].copy()
print('airplanes (all years):', len(df))

airplanes (all years): 77905


In [7]:
#### Amateur.Built
The client wants professional builds only. We keep 'No'. Rows with a missing value cannot be confirmed as professional builds, so we drop them.

SyntaxError: invalid syntax (2999390516.py, line 2)

In [ ]:
print(df['Amateur.Built'].value_counts(dropna=False))
df = df[df['Amateur.Built'] == 'No'].copy()
print('professional-build airplanes:', len(df))

In [ ]:
#### Event.Date
The latest event is Dec 2022 and we assume a 40-year maximum lifetime, so a make/model involved in an accident before 1983 is assumed retired. We keep events from 1 Jan 1983 onwards.

In [ ]:
df = df[df['Event.Date'] >= '1983-01-01'].copy()
print('rows after filtering:', len(df))
print(df['Event.Date'].dt.year.value_counts().sort_index().head(3))

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [ ]:
**Cleaning assumptions - injuries**
1. Total.Fatal.Injuries is sometimes blank although Injury.Severity says e.g. Fatal(3); we recover the count from that text.
2. For the four count columns (Fatal / Serious / Minor / Uninjured), a blank next to other filled-in counts means none, so we fill with 0. If all four are blank we have no information about the occupants, so those rows are dropped.
3. Total.Passengers = Fatal + Serious + Minor + Uninjured (everyone aboard, crew included, the only occupancy estimate in the data). Rows where this is 0 are dropped.
4. Fatal.Serious.Fraction = (Fatal + Serious) / Total.Passengers, a number between 0 and 1.

In [ ]:
inj_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']

# 1. recover fatal count from the Injury.Severity text, e.g. 'Fatal(3)' -> 3
fatal_from_text = df['Injury.Severity'].str.extract(r'Fatal\((\d+)\)')[0].astype(float)
df['Total.Fatal.Injuries'] = df['Total.Fatal.Injuries'].fillna(fatal_from_text)

# 2. drop rows with no occupant information at all, then fill remaining blanks with 0
df = df.dropna(subset=inj_cols, how='all').copy()
df[inj_cols] = df[inj_cols].fillna(0)

# 3. + 4. derived measures
df['Total.Passengers'] = df[inj_cols].sum(axis=1)
df = df[df['Total.Passengers'] > 0].copy()
df['Fatal.Serious.Fraction'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.Passengers']

print(len(df), 'rows')
df[['Total.Passengers', 'Fatal.Serious.Fraction']].describe()

In [ ]:
assert df['Fatal.Serious.Fraction'].between(0, 1).all()
df.nlargest(5, 'Total.Passengers')[['Make', 'Model', 'Event.Date', 'Total.Passengers', 'Fatal.Serious.Fraction']]

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [ ]:
print(df['Aircraft.damage'].value_counts(dropna=False))

In [ ]:
**Aircraft.damage cleaning**
- `Unknown` is a placeholder for missing -> NaN.
- Rows with no damage information cannot contribute to a destruction rate, so they are dropped (a small share of the data).
- **Destroyed** = 1 if damage == `Destroyed`, else 0 (Substantial and Minor both count as "survived").

In [ ]:
df['Aircraft.damage'] = df['Aircraft.damage'].replace('Unknown', np.nan)
df = df.dropna(subset=['Aircraft.damage']).copy()
df['Destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(int)

print(len(df), 'rows')
df['Aircraft.damage'].value_counts()

In [ ]:
df['Destroyed'].mean()

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [ ]:
**Make - cleaning tasks**
1. Drop rows with a missing Make.
2. Upper-case, strip whitespace, remove punctuation and collapse repeated spaces (`Cessna`, `CESSNA `, `cessna`).
3. Merge spelling / corporate variants of the same manufacturer (e.g. `CESSNA AIRCRAFT CO`, `HAWKER BEECHCRAFT`, `AIRBUS INDUSTRIE`, `DEHAVILLAND`, `GATES LEARJET`...).
4. Keep only makes with **at least 50 records**, so that comparisons between makes rest on enough data.

In [ ]:
df = df.dropna(subset=['Make']).copy()
print('unique makes before:', df['Make'].nunique())

df['Make'] = (df['Make'].str.upper()
              .str.replace(r'[^\w\s]', ' ', regex=True)   # punctuation -> space
              .str.replace(r'\s+', ' ', regex=True)        # collapse spaces
              .str.strip())
print('unique makes after basic tidy-up:', df['Make'].nunique())
df['Make'].value_counts().head(40)

In [8]:
# merge variants of the same manufacturer (regex on the start of the name -> canonical name)
make_rules = {
    r'^CESSNA.*': 'CESSNA',
    r'^(NEW )?PIPER.*': 'PIPER',
    r'^(HAWKER )?BEECH.*': 'BEECH',
    r'^(THE )?BOEING( COMPANY)?$': 'BOEING',
    r'^AIRBUS( INDUSTRIES?)?$': 'AIRBUS',
    r'^MCDONNELL DOUGLAS( AIRCRAFT)?( CO(RP(ORATION)?)?)?$': 'MCDONNELL DOUGLAS',
    r'^DE ?HAVILLAND.*': 'DE HAVILLAND',
    r'^CIRRUS.*': 'CIRRUS',
    r'^MOONEY.*': 'MOONEY',
    r'^EMBRAER.*': 'EMBRAER',
    r'^BOMBARDIER.*': 'BOMBARDIER',
    r'^(GATES )?LEARJET.*': 'LEARJET',
    r'^AIR TRACTOR.*': 'AIR TRACTOR',
    r'^AVIAT.*': 'AVIAT',
    r'^RAYTHEON.*': 'RAYTHEON',
    r'^ROCKWELL.*': 'ROCKWELL',
    r'^DIAMOND.*': 'DIAMOND',
    r'^SAAB.*': 'SAAB',
    r'^GRUMMAN AMERICAN.*': 'GRUMMAN AMERICAN',
    r'^GRUMMAN (ACFT ENG COR )?SCHWEIZER$': 'SCHWEIZER',
    r'^AMERICAN CHAMPION.*': 'CHAMPION',
    r'^MAULE.*': 'MAULE',
    r'^AERO COMMANDER.*': 'AERO COMMANDER',
    r'^FAIRCHILD.*': 'FAIRCHILD',
    r'^BELLANCA.*': 'BELLANCA',
    r'^ERCOUPE.*': 'ERCOUPE',
    r'^GULFSTREAM.*': 'GULFSTREAM',
    r'^BRITISH AEROSPACE.*': 'BRITISH AEROSPACE',
    r'^SWEARINGEN.*': 'SWEARINGEN',
    r'^CANADAIR.*': 'CANADAIR',
    r'^MITSUBISHI.*': 'MITSUBISHI',
}
for pattern, canonical in make_rules.items():
    df['Make'] = df['Make'].str.replace(pattern, canonical, regex=True)

print('unique makes after merging variants:', df['Make'].nunique())

unique makes after merging variants: 7202


In [9]:
make_counts = df['Make'].value_counts()
keep_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(keep_makes)].copy()
print(len(keep_makes), 'makes kept ->', len(df), 'rows')
df['Make'].value_counts().head(60)

90 makes kept -> 65482 rows


Make
Cessna                            22226
Piper                             12028
CESSNA                             4972
Beech                              4328
PIPER                              2895
Boeing                             1587
BOEING                             1166
Grumman                            1093
Mooney                             1092
BEECH                              1078
Bellanca                            886
Air Tractor                         590
Aeronca                             487
Maule                               445
Champion                            428
Mcdonnell Douglas                   408
De Havilland                        382
Aero Commander                      360
Stinson                             348
Rockwell                            331
Taylorcraft                         321
Luscombe                            319
AIR TRACTOR                         314
CIRRUS                              306
North American                     

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [10]:
# 1. get rid of NaNs
df = df.dropna(subset=['Model']).copy()

# 2. tidy: upper-case, strip, collapse repeated spaces
df['Model'] = df['Model'].str.upper().str.strip().str.replace(r'\s+', ' ', regex=True)

print('unique models:', df['Model'].nunique())
df['Model'].value_counts().head(15)

unique models: 4855


Model
152          2367
172          1754
172N         1163
PA-28-140     932
150           829
172M          798
172P          689
182           659
180           622
150M          585
PA-18         578
PA-18-150     574
PA-28-180     572
PA-28-161     568
PA-28-181     532
Name: count, dtype: int64

In [11]:
makes_per_model = df.groupby('Model')['Make'].nunique()
print('model labels used by more than one make:', (makes_per_model > 1).sum(), 'of', len(makes_per_model))
makes_per_model[makes_per_model > 1].sort_values(ascending=False).head(10)

model labels used by more than one make: 1178 of 4855


Model
G-164B    9
G-164A    8
AA-5B     8
500       7
AA-5A     7
G-164     7
690       6
G-164C    6
100       6
112       6
Name: Make, dtype: int64

In [12]:
df[df['Model'] == '500']['Make'].value_counts()

Make
Cessna            21
Aero Commander    14
CESSNA             6
AERO COMMANDER     4
Rockwell           2
North American     1
EMBRAER            1
Name: count, dtype: int64

In [13]:
**Result:** Model labels are *not* unique to a make (e.g. `500` appears under several different manufacturers, and labels such as `G-164` appear under several spellings/subsidiaries of the same company), so grouping by `Model` alone would mix different aircraft. We create a unique identifier **`Plane.Type` = Make + Model** and use that for all model-level analysis.

SyntaxError: invalid syntax (3495638308.py, line 1)

In [14]:
df['Plane.Type'] = df['Make'] + ' ' + df['Model']
print('unique plane types:', df['Plane.Type'].nunique())
df['Plane.Type'].value_counts().head(10)

unique plane types: 6294


Plane.Type
Cessna 152         2168
Cessna 172         1254
Cessna 172N         996
Piper PA-28-140     812
Cessna 150          716
Cessna 172M         667
Cessna 172P         597
Cessna 150M         539
Piper PA-18         539
Piper PA-28-161     502
Name: count, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [15]:
for col in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    print('-----', col)
    print(df[col].value_counts(dropna=False))

----- Engine.Type
Engine.Type
Reciprocating      54964
NaN                 4336
Turbo Prop          2783
Turbo Fan           2196
Unknown              625
Turbo Jet            544
Geared Turbofan       11
UNK                    1
Name: count, dtype: int64
----- Weather.Condition
Weather.Condition
VMC    56154
IMC     5238
NaN     3152
UNK      722
Unk      194
Name: count, dtype: int64
----- Number.of.Engines
Number.of.Engines
1.0    51334
2.0     9588
NaN     3561
3.0      447
4.0      396
0.0      134
Name: count, dtype: int64
----- Purpose.of.flight
Purpose.of.flight
Personal                     36816
Instructional                 8454
Unknown                       5114
NaN                           4489
Aerial Application            3486
Business                      3254
Positioning                   1013
Ferry                          616
Other Work Use                 532
Executive/corporate            401
Aerial Observation             389
Public Aircraft                329
Sky

In [16]:
# ---- Engine.Type ----
df['Engine.Type'] = df['Engine.Type'].replace({'Unknown': np.nan, 'UNK': np.nan, 'NONE': np.nan, 'LR': np.nan,
                                               'Geared Turbofan': 'Turbo Fan'})
et = df['Engine.Type'].value_counts()
df.loc[df['Engine.Type'].isin(et[et < 30].index), 'Engine.Type'] = np.nan

# ---- Weather.Condition ----
df['Weather.Condition'] = df['Weather.Condition'].str.upper().replace({'UNK': np.nan, 'UNKNOWN': np.nan})

# ---- Number.of.Engines ----
df.loc[df['Number.of.Engines'] == 0, 'Number.of.Engines'] = np.nan
ne = df['Number.of.Engines'].value_counts()
df.loc[df['Number.of.Engines'].isin(ne[ne < 30].index), 'Number.of.Engines'] = np.nan

# ---- Purpose.of.flight ----
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace('Unknown', np.nan)
pf = df['Purpose.of.flight'].value_counts()
df.loc[df['Purpose.of.flight'].isin(pf[pf < 100].index), 'Purpose.of.flight'] = 'Other'

# ---- Broad.phase.of.flight ----
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].replace('Unknown', np.nan)

for col in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    print('-----', col)
    print(df[col].value_counts(dropna=False))

----- Engine.Type
Engine.Type
Reciprocating    54964
NaN               4962
Turbo Prop        2783
Turbo Fan         2207
Turbo Jet          544
Name: count, dtype: int64
----- Weather.Condition
Weather.Condition
VMC    56154
IMC     5238
NaN     4068
Name: count, dtype: int64
----- Number.of.Engines
Number.of.Engines
1.0    51334
2.0     9588
NaN     3695
3.0      447
4.0      396
Name: count, dtype: int64
----- Purpose.of.flight
Purpose.of.flight
Personal               36816
NaN                     9603
Instructional           8454
Aerial Application      3486
Business                3254
Positioning             1013
Ferry                    616
Other Work Use           532
Executive/corporate      401
Aerial Observation       389
Public Aircraft          329
Other                    268
Skydiving                174
Flight Test              125
Name: count, dtype: int64
----- Broad.phase.of.flight
Broad.phase.of.flight
NaN            18092
Landing        12914
Takeoff         9715
Cr

In [17]:
print([c for c in df.columns if 'Total' in c or 'Fraction' in c or c == 'Destroyed'])

['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']


In [18]:
inj_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']

# 1. recover fatal count from the Injury.Severity text, e.g. 'Fatal(3)' -> 3
fatal_from_text = df['Injury.Severity'].str.extract(r'Fatal\((\d+)\)')[0].astype(float)
df['Total.Fatal.Injuries'] = df['Total.Fatal.Injuries'].fillna(fatal_from_text)

# 2. drop rows with no occupant information at all, then fill remaining blanks with 0
df = df.dropna(subset=inj_cols, how='all').copy()
df[inj_cols] = df[inj_cols].fillna(0)

# 3. + 4. derived measures
df['Total.Passengers'] = df[inj_cols].sum(axis=1)
df = df[df['Total.Passengers'] > 0].copy()
df['Fatal.Serious.Fraction'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.Passengers']

print(len(df), 'rows')
df[['Total.Passengers', 'Fatal.Serious.Fraction']].describe()

64415 rows


,Total.Passengers,Fatal.Serious.Fraction
count,64415.000000,64415.000000
mean,7.446837,0.264364
std,32.042110,0.425688
min,1.000000,0.000000
25%,1.000000,0.000000
50%,2.000000,0.000000
75%,3.000000,0.500000
max,699.000000,1.000000


In [19]:
df[['Number.of.Engines', 'Total.Passengers', 'Fatal.Serious.Fraction']].describe()

,Number.of.Engines,Total.Passengers,Fatal.Serious.Fraction
count,61492.000000,64415.000000,64415.000000
mean,1.185325,7.446837,0.264364
std,0.449604,32.042110,0.425688
min,1.000000,1.000000,0.000000
25%,1.000000,1.000000,0.000000
50%,1.000000,2.000000,0.000000
75%,1.000000,3.000000,0.500000
max,4.000000,699.000000,1.000000


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [ ]:
The lab rule: **keep columns with more than 20,000 non-null values**.

In [20]:
print('rows:', len(df))
non_null = df.notna().sum().sort_values()
non_null

rows: 64415


Schedule                   9023
Air.carrier               10940
FAR.Description           20687
Longitude                 22297
Latitude                  22301
Airport.Code              38624
Airport.Name              40758
Broad.phase.of.flight     47359
Publication.Date          53671
Purpose.of.flight         55811
Engine.Type               60314
Report.Status             60895
Weather.Condition         61306
Number.of.Engines         61492
Aircraft.damage           62517
Registration.Number       63657
Country                   64314
Amateur.Built             64366
Location                  64383
Total.Uninjured           64415
_make                     64415
_key                      64415
Plane.Type                64415
Total.Minor.Injuries      64415
Event.Id                  64415
Total.Fatal.Injuries      64415
Total.Passengers          64415
Model                     64415
Make                      64415
Aircraft.Category         64415
Injury.Severity           64415
Event.Da

In [21]:
# drop every column with <= 20,000 non-nulls
before = df.shape[1]
df = df.dropna(axis=1, thresh=20001)
print('dropped', before - df.shape[1], 'columns; remaining:', df.shape)
list(df.columns)

dropped 2 columns; remaining: (64415, 34)


['Event.Id',
 'Investigation.Type',
 'Accident.Number',
 'Event.Date',
 'Location',
 'Country',
 'Latitude',
 'Longitude',
 'Airport.Code',
 'Airport.Name',
 'Injury.Severity',
 'Aircraft.damage',
 'Aircraft.Category',
 'Registration.Number',
 'Make',
 'Model',
 'Amateur.Built',
 'Number.of.Engines',
 'Engine.Type',
 'FAR.Description',
 'Purpose.of.flight',
 'Total.Fatal.Injuries',
 'Total.Serious.Injuries',
 'Total.Minor.Injuries',
 'Total.Uninjured',
 'Weather.Condition',
 'Broad.phase.of.flight',
 'Report.Status',
 'Publication.Date',
 '_make',
 '_key',
 'Plane.Type',
 'Total.Passengers',
 'Fatal.Serious.Fraction']

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [22]:
df.to_csv('data/cleaned_aviation_data.csv', index=False)
print(pd.read_csv('data/cleaned_aviation_data.csv', low_memory=False).shape)

(64415, 34)
